# 1. Check the files

What files do we have, and what is inside them? This notebook reads file information. It does not download or remove data.

Choose the **PPP first look** kernel, then run the cells in order.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "analysis/__init__.py").is_file())
sys.path.insert(0, str(PROJECT))
from analysis.data.schema_inventory import (
    load_file_list, folder_summary, inspect_samples, save_inventory,
)

manifest, data_root, manifest_path, output, mode = load_file_list(PROJECT)
print("Data folder:", data_root)
print("Saved file list:", manifest_path)

## Files by folder

These counts come from the saved download list, not a fresh CERN scan. Each row counts files directly in that folder; subfolders have their own rows.

For a quick check, we open the smallest nonempty ROOT file in each folder. Other files may differ.

In [ ]:
folders, samples = folder_summary(manifest)
display(pd.DataFrame(folders).round({"GB": 3}))
print(f"{len(manifest['files'])} files in the list; {len(samples)} ROOT files to inspect.")

## What is inside?

A **tree** is like a table, a **branch** is a column, and an **entry** is a row. A row can hold several jets and their cells.

We check file sizes and read the tree names, row counts and variable names. This does not recheck file hashes. Download dates can differ from source dates.

In [ ]:
inventory = inspect_samples(PROJECT, data_root, manifest_path, manifest, samples, mode)
save_inventory(inventory, output)

for sample in inventory["samples"]:
    print(f"\n{sample['group']} ({sample['status']})")
    for item in sample["schema"]["objects"]:
        if "entries" in item:
            branches = list(item.get("branches", {}))
            print(f"  {item['key']}: {item['entries']:,} rows, {len(branches)} variables")
            print("  Examples:", ", ".join(branches[:12]))
print("\nFull report:", output)

## What to take from this

The later notebooks use `JetContent` from LLP and unpaired files. ZMUMU uses a different format (AOD), so it is not part of this training.

Units are **MeV** for energy and momentum, **mm** for position and **ns** for time. An unpaired file is BIB-enriched; its name does not prove that every jet is BIB.

Still to confirm with Emil: the exact production version and which trigger cuts to use across runs.

Next: [look at a small sample](02_first_data_look.ipynb).